# StageOps: limpieza de `concerts.csv`

**Hackatón SO-2026-041 · Investigación de incidencias en la venta de entradas**

## Contexto
`concerts.csv` es el catálogo de la cartera revisada: **una fila por concierto** (seis en total). Aporta el contexto comercial de cada operación y, sobre todo, permite **asignar cada reserva, pago y entrada a un proveedor** (LumaTicket o EntradaLink), que es lo que habilita la comparación entre proveedores que pide el encargo.

Es un catálogo pequeño, así que aquí el objetivo no es limpiar mucho, sino **dejarlo verificado y listo para unir**: comprobar que coincide con la cartera del encargo, que sus fechas son coherentes y derivar las columnas que usarán los demás notebooks.

`CSV original → Inspección → Tipado → Validación (incluida la cartera del encargo) → Columnas derivadas → Tabla limpia`

El CSV original (`data/raw/`) no se modifica.

## 1. Método de trabajo

1. **Inspeccionar** el fichero tal como llega.
2. **Tipar** fechas (UTC) y números.
3. **Validar** formato, unicidad y coherencia temporal.
4. **Contrastar con el encargo**: la cartera del documento indica concierto, artista, recinto, ciudad y proveedor; el CSV debe coincidir.
5. **Derivar** las columnas útiles para la integración (horas locales, días de venta, proveedor).
6. **Validar con `assert`** y **exportar**.

Con seis filas, cualquier error se vería a simple vista; aun así se comprueba por código para que el resultado sea reproducible y no dependa de una revisión visual.

## 2. Importaciones y ubicación de los archivos

La raíz del repositorio se localiza subiendo desde la carpeta de trabajo hasta encontrar `data/raw`. Las rutas se definen una sola vez como constantes.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

CARPETA_BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'raw').is_dir())
RUTA_ORIGINAL = CARPETA_BASE / 'data' / 'raw' / 'concerts.csv'
CARPETA_LIMPIOS = CARPETA_BASE / 'data' / 'clean'
RUTA_LIMPIO = CARPETA_LIMPIOS / 'concerts_clean.csv'

SNAPSHOT_AT = pd.Timestamp('2026-10-28 23:59:59', tz='UTC')   # cierre de datos indicado en el encargo
FORMATO_FECHA = '%Y-%m-%dT%H:%M:%SZ'                          # ISO 8601 UTC
COLUMNAS_FECHA = ['sales_open_at', 'doors_open_at', 'concert_start_at', 'snapshot_at']

# Cartera según el documento de encargo (concierto, artista, recinto, ciudad, proveedor)
CARTERA_ENCARGO = pd.DataFrame([
    ('CON-01', 'Nébula Norte',    'Estadio del Litoral',           'Valencia', 'LumaTicket'),
    ('CON-02', 'Vera Sol',        'Estadio Metropolitano del Sur', 'Sevilla',  'EntradaLink'),
    ('CON-03', 'Atlas Eléctrico', 'Estadio Costa Verde',           'Bilbao',   'LumaTicket'),
    ('CON-04', 'Mar Abierto',     'Estadio del Turia Norte',       'Valencia', 'EntradaLink'),
    ('CON-05', 'Luna Valdés',     'Estadio Sierra Central',        'Madrid',   'LumaTicket'),
    ('CON-06', 'Los Días Azules', 'Estadio Bahía Atlántica',       'A Coruña', 'EntradaLink'),
], columns=['concert_id', 'artist', 'venue', 'city', 'ticketing_company_name']).set_index('concert_id')

pd.set_option('display.max_columns', None)

if not RUTA_ORIGINAL.exists():
    raise FileNotFoundError(f'No se encuentra {RUTA_ORIGINAL}. Revisa el directorio de trabajo: {Path.cwd()}')
CARPETA_LIMPIOS.mkdir(parents=True, exist_ok=True)
print('Carpeta base:', CARPETA_BASE)

## 3. Inspección del fichero original

Se lee con `encoding='utf-8-sig'` porque el CSV lleva BOM. Al ser tan pequeño se puede mostrar entero.

In [ ]:
df_raw = pd.read_csv(RUTA_ORIGINAL, encoding='utf-8-sig')

print(f'Registros: {len(df_raw)} | columnas: {list(df_raw.columns)}\n')
df_raw.info()
display(df_raw)
print('Valores ausentes por columna:', df_raw.isna().sum()[lambda s: s > 0].to_dict() or 'ninguno')

**Lo que muestran los datos:** seis conciertos, trece columnas y ningún valor ausente. Las fechas vienen como texto ISO 8601 (`...Z`) y `saleable_capacity` ya es numérico. `snapshot_at` es idéntico en todas las filas: es el cierre común de la exportación, no una propiedad de cada concierto. Los identificadores (`CON-01`…`CON-06`) no vienen ordenados, y el diccionario avisa de que no debe deducirse el orden temporal de la posición de las filas ni de los identificadores.

## 4. Tipado y normalización

Se trabaja sobre una copia para conservar `df_raw` intacto. Las fechas pasan a `datetime64` en **UTC**; con `errors='coerce'` una fecha ilegible se convertiría en `NaT` y la validación la contaría. Se eliminan espacios sobrantes en los textos.

In [ ]:
df_concerts = df_raw.copy()

for columna in df_concerts.select_dtypes(include=['object', 'string']).columns:
    df_concerts[columna] = df_concerts[columna].astype('string').str.strip()

for columna in COLUMNAS_FECHA:
    df_concerts[columna] = pd.to_datetime(df_concerts[columna], utc=True, errors='coerce')

df_concerts['saleable_capacity'] = pd.to_numeric(df_concerts['saleable_capacity'], errors='raise').astype('int64')
df_concerts = df_concerts.sort_values('concert_id').reset_index(drop=True)

df_concerts.dtypes.to_frame('tipo')

## 5. Validaciones de calidad

Cada comprobación cuenta cuántas filas la **incumplen**. Además de los formatos se verifica la lógica temporal del negocio: las ventas abren antes que las puertas, las puertas abren antes del inicio de la actuación, y los conciertos no se han celebrado todavía en el cierre de datos.

In [ ]:
from zoneinfo import available_timezones

comprobaciones = pd.Series({
    'concert_id duplicado': df_concerts['concert_id'].duplicated().sum(),
    'Formato de concert_id distinto de CON-NN': (~df_concerts['concert_id'].str.match(r'^CON-\d{2}$')).sum(),
    'Fechas no interpretables': df_concerts[COLUMNAS_FECHA].isna().any(axis=1).sum(),
    'Capacidad no positiva': (df_concerts['saleable_capacity'] <= 0).sum(),
    'Ventas no abren antes que las puertas': (~(df_concerts['sales_open_at'] < df_concerts['doors_open_at'])).sum(),
    'Puertas no abren antes del inicio': (~(df_concerts['doors_open_at'] < df_concerts['concert_start_at'])).sum(),
    'Concierto ya celebrado en el cierre de datos': (df_concerts['concert_start_at'] <= SNAPSHOT_AT).sum(),
    'Ventas abiertas después del cierre de datos': (df_concerts['sales_open_at'] > SNAPSHOT_AT).sum(),
    'snapshot_at distinto del cierre del encargo': (df_concerts['snapshot_at'] != SNAPSHOT_AT).sum(),
    'Zona horaria no válida': (~df_concerts['timezone'].isin(available_timezones())).sum(),
    'Mismo recinto y fecha repetidos': df_concerts.duplicated(['venue', 'concert_start_at']).sum(),
}, name='filas que incumplen')

display(comprobaciones.to_frame())

# El identificador del proveedor y su nombre deben corresponderse uno a uno
print('Proveedores (id -> nombre):')
display(df_concerts.groupby('ticketing_company_id')['ticketing_company_name'].agg(['nunique', 'first']))

**Resultado de la calidad:** todas las reglas se cumplen y la relación identificador-nombre de proveedor es 1:1 (`LT` = LumaTicket, `EL` = EntradaLink). Las ventas abren entre el 2 y el 23 de octubre de 2026 y las actuaciones están previstas entre febrero y marzo de 2027, así que a fecha de cierre ninguna se ha celebrado.

## 6. Contraste con la cartera del encargo

El documento de encargo lista los seis conciertos con su artista, recinto, ciudad y proveedor. Si el CSV difiriera en algo, el análisis atribuiría operaciones al proveedor equivocado, y toda la comparación entre proveedores quedaría invalidada. Se compara campo a campo.

In [ ]:
catalogo = df_concerts.set_index('concert_id')[CARTERA_ENCARGO.columns]

print('Conciertos del encargo ausentes en el CSV:', sorted(set(CARTERA_ENCARGO.index) - set(catalogo.index)) or 'ninguno')
print('Conciertos del CSV que no están en el encargo:', sorted(set(catalogo.index) - set(CARTERA_ENCARGO.index)) or 'ninguno')

diferencias = (catalogo.astype(str) != CARTERA_ENCARGO.loc[catalogo.index].astype(str))
print('Celdas que difieren de la cartera del encargo:', int(diferencias.to_numpy().sum()))
display(diferencias.sum().to_frame('diferencias'))

**Resultado:** el catálogo coincide exactamente con la cartera del encargo: mismos seis conciertos y ninguna diferencia en artista, recinto, ciudad o proveedor. Cada proveedor gestiona tres conciertos.

## 7. Columnas derivadas

Se añaden columnas que los demás notebooks necesitarán y que conviene calcular **una sola vez**, con la misma definición para todos:

| Columna | Definición | Para qué |
| :--- | :--- | :--- |
| `sales_open_local`, `doors_open_local`, `concert_start_local` | las fechas convertidas a la zona `timezone` del recinto | presentar horas locales en el informe |
| `days_on_sale_at_snapshot` | días entre `sales_open_at` y el cierre de datos | comparar la antigüedad de la venta |
| `days_to_concert_at_snapshot` | días entre el cierre de datos y `concert_start_at` | contexto temporal |

Las columnas en UTC **no se modifican**: la comparación de relojes (reserva, pago, emisión) se hace siempre en UTC, como indica el diccionario.

In [ ]:
df_limpio = df_concerts.copy()

# Una sola zona horaria en este catálogo; si hubiera varias, se convertiría fila a fila
assert df_limpio['timezone'].nunique() == 1, 'hay varias zonas horarias: convertir por fila'
zona = df_limpio['timezone'].iloc[0]

for utc, local in [('sales_open_at', 'sales_open_local'), ('doors_open_at', 'doors_open_local'), ('concert_start_at', 'concert_start_local')]:
    df_limpio[local] = df_limpio[utc].dt.tz_convert(zona).dt.strftime('%Y-%m-%d %H:%M')

df_limpio['days_on_sale_at_snapshot'] = ((SNAPSHOT_AT - df_limpio['sales_open_at']).dt.total_seconds() / 86400).round(1)
df_limpio['days_to_concert_at_snapshot'] = ((df_limpio['concert_start_at'] - SNAPSHOT_AT).dt.total_seconds() / 86400).round(1)

display(df_limpio[['concert_id', 'artist', 'ticketing_company_id', 'saleable_capacity',
                   'sales_open_local', 'doors_open_local', 'concert_start_local',
                   'days_on_sale_at_snapshot', 'days_to_concert_at_snapshot']])

**Lo que muestran las columnas derivadas:** las ventas abrieron a las 10:00 UTC (12:00 hora local, porque en octubre rige el horario de verano) y las puertas abren a las 18:00 UTC, que son las 19:00 locales en febrero y marzo. Los conciertos con más días de venta (CON-01 y CON-02, abiertos entre el 2 y el 3 de octubre) tendrán más reservas que los que abrieron el 23 de octubre; esa diferencia de exposición hay que tenerla presente al comparar conciertos, para no confundir menos ventas con menos incidencias.

## 8. Visualización de control

Gráfica de control, no de informe: capacidad y días de venta por concierto, coloreados por proveedor.

In [ ]:
colores = {'LT': '#2f6db5', 'EL': '#d98324'}
fig, ejes = plt.subplots(1, 2, figsize=(12, 3.8))

ejes[0].bar(df_limpio['concert_id'], df_limpio['saleable_capacity'], color=df_limpio['ticketing_company_id'].map(colores))
ejes[0].set_title('Aforo comercial (saleable_capacity)')
ejes[0].set_ylabel('localidades')

ejes[1].bar(df_limpio['concert_id'], df_limpio['days_on_sale_at_snapshot'], color=df_limpio['ticketing_company_id'].map(colores))
ejes[1].set_title('Días de venta al cierre de datos')
ejes[1].set_ylabel('días')

ejes[0].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in colores.values()], labels=['LumaTicket', 'EntradaLink'])
plt.tight_layout()
plt.show()

## 9. Validación y exportación

Antes de exportar se comprueban con `assert` las propiedades que necesitarán los demás notebooks para poder unir con este catálogo: clave única, un solo proveedor por concierto y fechas coherentes. Después se guarda `concerts_clean.csv` con las fechas en ISO 8601 UTC y se relee para comprobar que conserva filas y columnas.

In [ ]:
def validar_catalogo(df):
    """Se comprueba que el catálogo es usable como tabla de referencia; cada assert detiene el proceso si falla."""
    def comprobar(condicion, descripcion):
        assert condicion, f'FALLO: {descripcion}'
        print(f'  ok · {descripcion}')

    comprobar(len(df) == 6, 'hay seis conciertos, como en el encargo')
    comprobar(df['concert_id'].is_unique, 'concert_id es único (clave primaria)')
    comprobar(df.groupby('concert_id')['ticketing_company_id'].nunique().eq(1).all(), 'cada concierto tiene un solo proveedor')
    comprobar(df.groupby('ticketing_company_id')['ticketing_company_name'].nunique().eq(1).all(), 'id y nombre de proveedor se corresponden 1:1')
    comprobar((df['sales_open_at'] < df['doors_open_at']).all() and (df['doors_open_at'] < df['concert_start_at']).all(), 'ventas < puertas < inicio en todos los conciertos')
    comprobar((df['saleable_capacity'] > 0).all(), 'aforo comercial positivo')
    comprobar(df.notna().all().all(), 'sin valores ausentes')


validar_catalogo(df_limpio)

salida = df_limpio.copy()
for columna in salida.select_dtypes(include='datetimetz').columns:
    salida[columna] = salida[columna].dt.strftime(FORMATO_FECHA)
salida.to_csv(RUTA_LIMPIO, index=False, encoding='utf-8')

relectura = pd.read_csv(RUTA_LIMPIO)
assert relectura.shape == salida.shape and list(relectura.columns) == list(salida.columns), 'el fichero cambió al exportar'
print(f'\n{RUTA_LIMPIO.name}: {relectura.shape[0]} filas · {relectura.shape[1]} columnas')

## 10. Reflexión final

**Qué se ha demostrado**
- El catálogo tiene seis conciertos, sin nulos ni duplicados, y coincide campo a campo con la cartera del encargo.
- Cada concierto tiene un único proveedor: LumaTicket gestiona CON-01, CON-03 y CON-05; EntradaLink, CON-02, CON-04 y CON-06.
- Las fechas son coherentes y ninguna actuación se ha celebrado en el cierre de datos.

**Cómo usarlo en la integración** (detalle en [docs/concerts_conclusiones.md](../docs/concerts_conclusiones.md))
- Es la tabla que **enlaza con el proveedor**: `reservations.concert_id` → `concerts.concert_id` da `ticketing_company_id`, y con él se une `system_changes`.
- La relación `concerts` → `reservations` es 1:N; usar `validate='many_to_one'` al unir desde reservas.
- `saleable_capacity` es el aforo **comercial** del montaje, no el máximo físico del estadio.

**Pendiente para la integración** (requiere otros ficheros)
- Reservas anteriores a `sales_open_at` de su concierto: serían registros incoherentes.
- Entradas vigentes de un concierto por encima de `saleable_capacity`.
- Reservas con `concert_id` que no existe en este catálogo.